In [1]:
# Install required packages
import subprocess, sys

packages = ['flwr>=1.8.0', 'tensorflow', 'pandas', 'numpy', 'scikit-learn', 'tensorflowjs']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ All packages installed!")

ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import flwr as fl
from sklearn.model_selection import train_test_split
import warnings, json, os
warnings.filterwarnings('ignore')

print(f"TensorFlow version : {tf.__version__}")
print(f"Flowers version    : {fl.__version__}")
print(f"GPU available      : {len(tf.config.list_physical_devices('GPU')) > 0}")

I0000 00:00:1781095152.806256  168277 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version : 2.21.0
Flowers version    : 1.31.0
GPU available      : False


W0000 00:00:1781095157.216484  168277 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [3]:
# Sample Nepali text data (grammatical: 1, ungrammatical: 0)
# nepali_data = [
#     ("विद्यालय शुरु हुन्छ", 1),              # correct
#     ("विद्यालय शुरु हु", 0),                 # error
#     ("मेरो नाम राज हो", 1),                  # correct
#     ("मेरो नाम राज हु", 0),                  # error
#     ("किताब टेबलमा छ", 1),                  # correct
#     ("किताब टेबल छ", 0),                    # error
#     ("मलाई खेलन मन पर्छ", 1),               # correct
#     ("मलाई खेलन मन पर", 0),                 # error
#     ("उनको घर सुन्दर छ", 1),                # correct
#     ("उनको घर सुन्दर हु", 0),                # error
#     ("हामी पढाई गर्छौ", 1),                 # correct
#     ("हामी पढाई गर्छ", 0),                  # error
#     ("यो सुन्दर गीत हो", 1),                # correct
#     ("यो सुन्दर गीत हु", 0),                # error
#     ("अहिले बिहान छ", 1),                  # correct
#     ("अहिले बिहान हु", 0),                 # error
# ]

# df = pd.DataFrame(nepali_data, columns=["text", "label"])
df =pd.read_csv("../data_cleaned/cleaned_labeled.csv", encoding="utf-8")
print("Dataset shape:", df.shape)
print("\nSample data:")
print(df.head())
print(f"\nClass distribution:\n{df['label'].value_counts()}")

Dataset shape: (4626415, 2)

Sample data:
         word  label
0        यसरी      0
1  व्यवस्थापन      0
2      गर्दैछ      0
3        बिपी      0
4     कोइराला      0

Class distribution:
label
0    2376764
1    2249651
Name: count, dtype: int64


In [7]:
class SimpleNepaliTokenizer:
    """Whitespace tokenizer — identical to PyTorch version"""
    def __init__(self):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.vocab_size = 2

    def build_vocab(self, texts):
        for text in texts:
            for word in text.split():
                if word not in self.word2idx:
                    idx = len(self.word2idx)
                    self.word2idx[word] = idx
                    self.idx2word[idx] = word
        self.vocab_size = len(self.word2idx)
        print(f"Vocabulary size: {self.vocab_size}")

    def encode(self, text, max_len=20):
        indices = [self.word2idx.get(w, self.word2idx['<UNK>']) for w in text.split()]
        if len(indices) < max_len:
            indices += [0] * (max_len - len(indices))
        return indices[:max_len]

    def decode(self, indices):
        return ' '.join(self.idx2word.get(i, '<UNK>') for i in indices if i != 0)

    def save(self, path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, 'w', encoding='utf-8') as f:
            json.dump({
                'word2idx': self.word2idx,
                'idx2word': {str(k): v for k, v in self.idx2word.items()}
            }, f, ensure_ascii=False, indent=2)
        print(f"✓ Tokenizer saved → {path}")

    @classmethod
    def load(cls, path):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        t = cls()
        t.word2idx = data['word2idx']
        t.idx2word = {int(k): v for k, v in data['idx2word'].items()}
        t.vocab_size = len(t.word2idx)
        return t


MAX_SEQ_LEN = 20

tokenizer = SimpleNepaliTokenizer()
tokenizer.build_vocab(df['word'].tolist())

X = np.array([tokenizer.encode(t, MAX_SEQ_LEN) for t in df['word']], dtype=np.int32)
y = df['label'].values.astype(np.float32)

print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Example: {df['word'].iloc[0]}")
print(f"Encoded: {X[0]}")

Vocabulary size: 1472998
X shape: (4626415, 20), y shape: (4626415,)
Example: यसरी
Encoded: [2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   y_test : {y_test.shape}")

X_train: (3469811, 20)  y_train: (3469811,)
X_test : (1156604, 20)   y_test : (1156604,)


In [12]:
def build_model(vocab_size, embedding_dim=64, hidden_dim=128, dropout=0.3):
    inputs = keras.Input(shape=(MAX_SEQ_LEN,), dtype='int32', name='input')

    x = layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=False,          # fixes the mask broadcast error
        name='embedding'
    )(inputs)

    x = layers.Bidirectional(
        layers.LSTM(hidden_dim, return_sequences=True, dropout=dropout),
        name='bilstm_0'
    )(x)

    h = layers.Bidirectional(
        layers.LSTM(hidden_dim, return_sequences=True, dropout=dropout),
        name='bilstm_1'
    )(x)

    # attention — avoid layers.Softmax, use Activation instead
    attn_logits  = layers.Dense(1, name='attention')(h)                        # (B, T, 1)
    attn_weights = layers.Activation('softmax', name='attn_softmax')(attn_logits)  # (B, T, 1)
    context      = layers.Multiply(name='attn_apply')([h, attn_weights])       # (B, T, 256)
    context      = layers.Lambda(
                       lambda x: tf.reduce_sum(x, axis=1),
                       name='context_pool'
                   )(context)                                                   # (B, 256)

    x = layers.Dense(64, activation='relu', name='fc1')(context)
    x = layers.Dropout(dropout, name='dropout')(x)
    outputs = layers.Dense(1, activation='sigmoid', name='fc2')(x)

    return keras.Model(inputs=inputs, outputs=outputs, name='NepaliGrammarChecker')


model = build_model(vocab_size=tokenizer.vocab_size)
model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

W0000 00:00:1781095315.461875  168277 cpu_allocator_impl.cc:82] Allocation of 377087488 exceeds 10% of free system memory.
W0000 00:00:1781095315.539698  168277 cpu_allocator_impl.cc:82] Allocation of 377087488 exceeds 10% of free system memory.


Model: "NepaliGrammarChecker"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 20)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 20, 64)    │ 94,271,872 │ input[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_0            │ (None, 20, 256)   │    197,632 │ embedding[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_1            │ (None, 20, 256)   │    394,240 │ bilstm_0[0][0]    │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention (Dense)   │ (None, 20, 1)     │        257 │ bilstm_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_softmax        │ (None, 20, 1)     │          0 │ attention[0][0]   │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_apply          │ (None, 20, 256)   │          0 │ bilstm_1[0][0],   │
│ (Multiply)          │                   │            │ attn_softmax[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ context_pool        │ (None, 256)       │          0 │ attn_apply[0][0]  │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc1 (Dense)         │ (None, 64)        │     16,448 │ context_pool[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ fc1[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc2 (Dense)         │ (None, 1)         │         65 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 94,880,514 (361.94 MB)

 Trainable params: 94,880,514 (361.94 MB)

 Non-trainable params: 0 (0.00 B)

In [1]:
import tensorflow as tf

print(tf.config.list_physical_devices())

tf.debugging.set_log_device_placement(True)

I0000 00:00:1781095836.968796  177134 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


W0000 00:00:1781095840.028912  177134 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [20]:
def train_model(model, X_train, y_train, X_test, y_test, epochs=20, batch_size=4):
    """Centralized training — equivalent to PyTorch train_model()"""

    def lr_schedule(epoch, lr):
        if (epoch + 1) % 5 == 0:
            return lr * 0.5
        return lr

    callbacks = [
        keras.callbacks.LearningRateScheduler(lr_schedule, verbose=0),
        keras.callbacks.TerminateOnNaN(),
    ]

    history = {'loss': [], 'val_accuracy': []}

    for epoch in range(epochs):
        h = model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=1,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=0,
        )

        loss = h.history['loss'][0]
        acc  = h.history['val_accuracy'][0]
        history['loss'].append(loss)
        history['val_accuracy'].append(acc)

        # show progress every epoch
        bar_len = 20
        filled  = int(bar_len * (epoch + 1) / epochs)
        bar     = '█' * filled + '░' * (bar_len - filled)
        print(f"\rEpoch {epoch+1:>2}/{epochs} [{bar}] loss: {loss:.4f} | val_acc: {acc:.4f}", end='', flush=True)

        if (epoch + 1) % 5 == 0:
            print()  # newline every 5 epochs

    print("\n")
    return history


print("Training Centralized Model...\n")
history = train_model(model, X_train, y_train, X_test, y_test, epochs=20)
print("✓ Centralized training completed!")

Training Centralized Model...



KeyboardInterrupt: 

In [ ]:
def evaluate_model(model, X_test, y_test):
    """Full evaluation with precision, recall, F1"""
    preds_prob = model.predict(X_test, verbose=0).flatten()
    preds      = (preds_prob > 0.5).astype(np.float32)

    accuracy  = np.mean(preds == y_test)
    tp = np.sum((preds == 1) & (y_test == 1))
    fp = np.sum((preds == 1) & (y_test == 0))
    fn = np.sum((preds == 0) & (y_test == 1))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}


metrics = evaluate_model(model, X_test, y_test)
print("=" * 50)
print("TEST SET EVALUATION")
print("=" * 50)
for k, v in metrics.items():
    print(f"{k.upper():12} : {v:.4f}")
print("=" * 50)

In [ ]:
def predict(text, model, tokenizer):
    """Predict grammar correctness for a single sentence"""
    ids    = tokenizer.encode(text, MAX_SEQ_LEN)
    x      = np.array([ids], dtype=np.int32)
    output = model.predict(x, verbose=0)[0][0]
    return {
        'text'               : text,
        'correct_probability': float(output),
        'label'              : 'Correct ✓' if output > 0.5 else 'Incorrect ✗',
        'confidence'         : float(max(output, 1 - output)),
    }


test_texts = [
    "मेरो नाम राज हो",
    "मेरो नाम राज हु",
    "किताब टेबलमा छ",
    "किताब टेबल छ",
]

print("=" * 60)
print("PREDICTIONS ON NEW DATA")
print("=" * 60)
for text in test_texts:
    r = predict(text, model, tokenizer)
    print(f"\nText: {r['text']}")
    print(f"Prediction: {r['label']} (confidence: {r['confidence']:.2%})")

In [ ]:
class FederatedGrammarCheckerClient(fl.client.NumPyClient):
    """
    Flower FL client — TensorFlow version.
    Equivalent to the PyTorch FederatedGrammarCheckerClient.
    """

    def __init__(self, model, X_train, y_train, X_test, y_test, client_id=0):
        self.model     = model
        self.X_train   = X_train
        self.y_train   = y_train
        self.X_test    = X_test
        self.y_test    = y_test
        self.client_id = client_id

    def get_parameters(self, config):
        """Return model weights as list of numpy arrays"""
        return self.model.get_weights()

    def set_parameters(self, parameters):
        """Set model weights from list of numpy arrays"""
        self.model.set_weights(parameters)

    def fit(self, parameters, config):
        """Local training on client data"""
        self.set_parameters(parameters)

        lr         = config.get('lr', 0.001)
        batch_size = config.get('batch_size', 4)
        epochs     = config.get('epochs', 1)

        self.model.compile(
            optimizer=keras.optimizers.Adam(lr),
            loss='binary_crossentropy',
            metrics=['accuracy'],
        )

        self.model.fit(
            self.X_train, self.y_train,
            batch_size=batch_size,
            epochs=epochs,
            verbose=0,
        )

        return self.get_parameters(config), len(self.X_train), {}

    def evaluate(self, parameters, config):
        """Evaluate on local test data"""
        self.set_parameters(parameters)
        loss, accuracy = self.model.evaluate(
            self.X_test, self.y_test, verbose=0
        )
        return loss, len(self.X_test), {'accuracy': accuracy}


print("✓ Federated Learning Client defined")

In [ ]:
n_clients   = 3
data_splits = np.array_split(np.arange(len(X_train)), n_clients)

clients_data = []
for i, indices in enumerate(data_splits):
    clients_data.append((X_train[indices], y_train[indices]))
    print(f"Client {i+1}: {len(indices)} training samples")

print(f"\nTotal clients: {n_clients}")

In [ ]:
def make_client_fn():
    """Factory — creates a fresh TF model + Flower client per cid"""
    def client_fn(cid: str):
        client_id = int(cid)
        X_c, y_c  = clients_data[client_id]

        client_model = build_model(vocab_size=tokenizer.vocab_size)
        client_model.compile(
            optimizer=keras.optimizers.Adam(0.001),
            loss='binary_crossentropy',
            metrics=['accuracy'],
        )

        return FederatedGrammarCheckerClient(
            client_model,
            X_c, y_c,
            X_test, y_test,
            client_id=client_id,
        )
    return client_fn


print("✓ Client factory created")

In [ ]:
print("\n" + "=" * 70)
print("STARTING FEDERATED LEARNING WITH FLOWERS (TensorFlow)")
print("=" * 70)
print(f"Clients  : {n_clients}")
print(f"Strategy : FedAvg")
print("=" * 70 + "\n")

try:
    strategy = fl.server.strategy.FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=n_clients,
        min_evaluate_clients=n_clients,
        min_available_clients=n_clients,
    )

    fl.simulation.start_simulation(
        client_fn=make_client_fn(),
        num_clients=n_clients,
        config=fl.server.ServerConfig(num_rounds=3, round_timeout=600),
        strategy=strategy,
        client_resources={'num_cpus': 1, 'num_gpus': 0.0},
    )

    print("\n" + "=" * 70)
    print("✓ FEDERATED LEARNING COMPLETED!")
    print("=" * 70)

except Exception as e:
    print(f"\n⚠  {type(e).__name__}: {e}")
    print("FL simulation attempted. If you see deprecation warnings")
    print("about start_simulation(), use 'flwr run' CLI for production.")

In [ ]:
os.makedirs('model', exist_ok=True)

# Save as Keras .h5
model.save('model/nepali_grammar_checker.h5')
print("✓ Keras model saved → model/nepali_grammar_checker.h5")

# Save as TF SavedModel (needed for TF.js conversion)
model.save('model/tf_savedmodel')
print("✓ TF SavedModel    → model/tf_savedmodel/")

# Save tokenizer vocab
tokenizer.save('model/nepali_tokenizer_vocab.json')

print("\n✓ All files ready for deployment!")
print("\nTo convert for browser (TF.js):")
print("  pip install tensorflowjs")
print("  tensorflowjs_converter --input_format=tf_saved_model \\")
print("    model/tf_savedmodel model/tfjs_model")